## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import json
import os
import glob
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 200)

print('Libraries loaded OK')

Libraries loaded OK


## 1. Detect the file format

In [2]:
DATA_DIR = Path('../data/raw')

# List every file recursively
all_files = list(DATA_DIR.rglob('*.*'))
for f in sorted(all_files):
    size_mb = f.stat().st_size / 1_000_000
    print(f'{size_mb:8.2f} MB  {f}')

print(f'\nTotal files found: {len(all_files)}')

  378.57 MB  ..\data\raw\test_complete.jsonl
 5085.96 MB  ..\data\raw\train_complete.jsonl
  372.70 MB  ..\data\raw\valid_complete.jsonl

Total files found: 3


In [ ]:
# Auto-detect format and load
DATA_FILE = Path('../data/raw/train_complete.jsonl')

ext = DATA_FILE.suffix.lower()
print(f'Detected extension: {ext}')

if ext == '.csv':
    df = pd.read_csv(DATA_FILE)
elif ext == '.json':
    df = pd.read_json(DATA_FILE)
elif ext == '.jsonl':
    # JSON Lines - each row is a separate JSON object
    records = []
    with open(DATA_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    df = pd.DataFrame(records)
    if ext == '.parquet':
        df = pd.read_parquet(DATA_FILE)
else:
    raise ValueError(f'Unknown format: {ext}. Open the file manually to inspect it.')

print(f'Loaded {len(df):,} rows x {len(df.columns)} columns')

Detected extension: .jsonl


## 2. Basic shape and columns

In [4]:
print('=== SHAPE ===')
print(f'Rows: {len(df):,}')
print(f'Columns: {len(df.columns)}')

print('\n=== COLUMN NAMES ===')
for i, col in enumerate(df.columns):
    print(f'  [{i:02d}] {col}')

=== SHAPE ===
Rows: 141,259
Columns: 22

=== COLUMN NAMES ===
  [00] patch
  [01] oldf
  [02] hunk
  [03] msg
  [04] ids
  [05] repo
  [06] ghid
  [07] old
  [08] new
  [09] lang
  [10] review_id
  [11] file
  [12] user_id
  [13] user_login
  [14] created_at
  [15] name
  [16] repo_aco
  [17] sys_aco
  [18] pkg_aco
  [19] repo_rso
  [20] sys_rso
  [21] pkg_rso


In [5]:
print('=== DATA TYPES ===')
print(df.dtypes)

print('\n=== MEMORY USAGE ===')
mem_mb = df.memory_usage(deep=True).sum() / 1_000_000
print(f'{mem_mb:.1f} MB')

=== DATA TYPES ===
patch             str
oldf              str
hunk              str
msg               str
ids            object
repo              str
ghid            int64
old               str
new               str
lang              str
review_id       int64
file              str
user_id         int64
user_login        str
created_at      int64
name              str
repo_aco      float64
sys_aco       float64
pkg_aco       float64
repo_rso      float64
sys_rso       float64
pkg_rso       float64
dtype: object

=== MEMORY USAGE ===
4815.5 MB


## 3. First look at the data

In [6]:
# First 3 rows
df.head(3)

,patch,oldf,hunk,msg,ids,repo,ghid,old,new,lang,review_id,file,user_id,user_login,created_at,name,repo_aco,sys_aco,pkg_aco,repo_rso,sys_rso,pkg_rso
0,"@@ -48,23 +59,29 @@ bool TransformationAddGlobalVariable::IsApplicable(\n if (!pointer_type) {\n return false;...","// Copyright (c) 2019 Google LLC\n//\n// Licensed under the Apache License, Version 2.0 (the ""License"");\n// you may...","@@ -66,6 +66,9 @@ bool TransformationAddGlobalVariable::IsApplicable(\n if (message_.initializer_id()) {\n // ...",Maybe assert false,"[24794, 17d785e3dce428351b39d1290ec28a383342c8cb, 91569fc31a46c739ae5ccf18437d5b4782ca3c3a]",KhronosGroup/SPIRV-Tools,3277,if (message_.initializer_id()) {\n // An initializer is not allowed if the storage class is Workgroup.\n ...,if (message_.initializer_id()) {\n // An initializer is not allowed if the storage class is Workgroup.\n ...,cpp,404018015,source/fuzz/transformation_add_global_variable.cpp,5478283,paulthomson,1586172414000,Paul Thomson,0.006860,0.007571,0.010309,0.045916,0.055273,0.669903
1,"@@ -13,21 +13,17 @@\n \n public class OnThisDayActivity extends SingleFragmentActivity<OnThisDayFragment> {\n pu...",package org.wikipedia.feed.onthisday;\n\nimport android.content.Context;\nimport android.content.Intent;\n\nimport a...,"@@ -16,7 +16,7 @@ public class OnThisDayActivity extends SingleFragmentActivity<OnThisDayFragment>\n public stat...",Would it be better if add annotations to the parameters?,"[20887, 3e39aa16bb601c90e73b98249a9dcd77f17d511e, 9ee2d9cfbcafbce2b81669cc6adf3936e4b7b94f]",wikimedia/apps-android-wikipedia,1602,"public static final String YEAR = ""year"";\n public static final String WIKISITE = ""wikisite"";\n- public ...","public static final String YEAR = ""year"";\n public static final String WIKISITE = ""wikisite"";\n+ public ...",java,494570285,app/src/main/java/org/wikipedia/feed/onthisday/OnThisDayActivity.java,2435576,cooltey,1600976839000,Cooltey Feng,0.008198,0.010611,0.053333,0.195692,0.198473,0.431818
2,"@@ -250,13 +250,22 @@ func (c *twoPhaseCommitter) prewriteSingleBatch(bo *Backoffer, batch batchKeys)\n \tfor i, k :...","// Copyright 2016 PingCAP, Inc.\n//\n// Licensed under the Apache License, Version 2.0 (the ""License"");\n// you may ...","@@ -253,10 +253,8 @@ func (c *twoPhaseCommitter) prewriteSingleBatch(bo *Backoffer, batch batchKeys)\n \n \tskipChec...",We don't need to check not nil before try to assert to bool.,"[177560, 027f2c8b97ec3b3684069c85e602eb76e2559696, bdf35dc64d5fd2c9136900d2cdb7f78779ac05c8]",pingcap/tidb,2288,\tskipCheck := false\n \toptSkipCheck := c.txn.us.GetOption(kv.SkipCheckForWrite)\n-\tif optSkipCheck != nil {\n-\t...,"\tskipCheck := false\n \toptSkipCheck := c.txn.us.GetOption(kv.SkipCheckForWrite)\n+\tif skip, ok := optSkipCheck.(...",go,93408210,store/tikv/2pc.go,891222,coocood,1482314531000,Evan Zhou,0.000000,0.000000,0.000000,0.567753,0.515823,0.569231


In [7]:
# Print one full example as JSON — easiest way to see all fields
print(json.dumps(df.iloc[0].to_dict(), indent=2, default=str))

{
  "patch": "@@ -48,23 +59,29 @@ bool TransformationAddGlobalVariable::IsApplicable(\n   if (!pointer_type) {\n     return false;\n   }\n-  // ... with Private storage class.\n-  if (pointer_type->storage_class() != SpvStorageClassPrivate) {\n+  // ... with the right storage class.\n+  if (pointer_type->storage_class() != storage_class) {\n     return false;\n   }\n-  // The initializer id must be the id of a constant.  Check this with the\n-  // constant manager.\n-  auto constant_id = ir_context->get_constant_mgr()->GetConstantsFromIds(\n-      {message_.initializer_id()});\n-  if (constant_id.empty()) {\n-    return false;\n-  }\n-  assert(constant_id.size() == 1 &&\n-         \"We asked for the constant associated with a single id; we should \"\n-         \"get a single constant.\");\n-  // The type of the constant must match the pointee type of the pointer.\n-  if (pointer_type->pointee_type() != constant_id[0]->type()) {\n-    return false;\n+  if (message_.initializer_id()) {\n

In [8]:
# Print a few more examples to see variation
for i in [0, 1, 100, 1000]:
    print(f'\n{"="*60}')
    print(f'Example #{i}')
    print(f'{'='*60}')
    print(json.dumps(df.iloc[i].to_dict(), indent=2, default=str))


Example #0
{
  "patch": "@@ -48,23 +59,29 @@ bool TransformationAddGlobalVariable::IsApplicable(\n   if (!pointer_type) {\n     return false;\n   }\n-  // ... with Private storage class.\n-  if (pointer_type->storage_class() != SpvStorageClassPrivate) {\n+  // ... with the right storage class.\n+  if (pointer_type->storage_class() != storage_class) {\n     return false;\n   }\n-  // The initializer id must be the id of a constant.  Check this with the\n-  // constant manager.\n-  auto constant_id = ir_context->get_constant_mgr()->GetConstantsFromIds(\n-      {message_.initializer_id()});\n-  if (constant_id.empty()) {\n-    return false;\n-  }\n-  assert(constant_id.size() == 1 &&\n-         \"We asked for the constant associated with a single id; we should \"\n-         \"get a single constant.\");\n-  // The type of the constant must match the pointee type of the pointer.\n-  if (pointer_type->pointee_type() != constant_id[0]->type()) {\n-    return false;\n+  if (message_.initializ

## 4. Missing values

In [9]:
print('=== MISSING VALUES ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_%': missing_pct
}).sort_values('missing_%', ascending=False)

print(missing_df[missing_df['missing_count'] > 0])
print(f'\nColumns with NO missing values: {(missing == 0).sum()}')

=== MISSING VALUES ===
Empty DataFrame
Columns: [missing_count, missing_%]
Index: []

Columns with NO missing values: 22


## 5. Explore text columns (README content)

In [10]:
# Find columns that contain long text (likely README / PR body)
print('=== TEXT COLUMNS (avg length > 100 chars) ===')
for col in df.columns:
    if df[col].dtype == object:
        avg_len = df[col].dropna().astype(str).str.len().mean()
        max_len = df[col].dropna().astype(str).str.len().max()
        print(f'{col:40s}  avg={avg_len:8.0f}  max={max_len:8.0f}')

=== TEXT COLUMNS (avg length > 100 chars) ===
ids                                       avg=      95  max=      96


In [11]:
# Find README-related rows using file paths

readme_rows = df[
    df["file"].str.contains(
        "readme",
        case=False,
        na=False
    )
]

print(f"README-related rows found: {len(readme_rows)}")

if len(readme_rows) > 0:

    sample = readme_rows.iloc[0]

    print("\nRepository:")
    print(sample["repo"])

    print("\nFile path:")
    print(sample["file"])

    print("\nCommit message:")
    print(sample["msg"])

    print("\n--- PATCH PREVIEW ---")
    print(str(sample["patch"])[:1000])

else:
    print("No README-related rows found.")

README-related rows found: 10

Repository:
cosmos/cosmos-sdk

File path:
cmd/gaia/cli_test/README.md

Commit message:
testnets _do not_ use atoms but actually use an imaginary token named stake (aka this PR change should be reversed)

--- PATCH PREVIEW ---
@@ -85,7 +85,7 @@ Example:
 	)
 	cmd.Flags().String(
 		server.FlagMinGasPrices, fmt.Sprintf("0.000006%s", sdk.DefaultBondDenom),
-		"Minimum gas prices to accept for transactions; All fees in a tx must meet this minimum (e.g. 0.01photino,0.001stake)",
+		"Minimum gas prices to accept for transactions; All fees in a tx must meet this minimum (e.g. 0.01photino,0.001atom)",


In [12]:
# Count README updates

readme_count = len(readme_rows)
total_count = len(df)

percentage = (readme_count / total_count) * 100

print(f"Total rows: {total_count:,}")
print(f"README-related rows: {readme_count:,}")
print(f"Percentage: {percentage:.4f}%")

Total rows: 141,259
README-related rows: 10
Percentage: 0.0071%


README-related file changes are extremely rare in the dataset,
which aligns with the observations reported in the paper.

In [13]:
readme_rows["file"].value_counts().head(20)

file
datadog_checks_dev/datadog_checks/dev/tooling/commands/validate/readmes.py                       2
cmd/gaia/cli_test/README.md                                                                      1
generators/common/templates/README.md.ejs                                                        1
lib/theme_template/README.md.erb                                                                 1
plugin/metrics/README.md                                                                         1
shared/graphql/queries/thread/getThreadMessageConnection.js                                      1
streams/src/main/java/org/apache/kafka/streams/processor/internals/metrics/ThreadMetrics.java    1
src/Console/Command/ReadmeCommand.php                                                            1
generators/kubernetes/templates/README-KUBERNETES.md.ejs                                         1
Name: count, dtype: int64

## 6. README length distribution

In [14]:
# GROUP FILE CHANGES INTO PRs

pr_groups = df.groupby("ghid")

print("Unique PRs:", len(pr_groups))

Unique PRs: 25257


## 7. Repositories and PR metadata

In [ ]:
# DETECT README-RELATED PRs

readme_prs = []

for pr_id, group in pr_groups:

    files = group["file"].astype(str).tolist()

    has_readme = any(
        "readme" in f.lower()
        for f in files
    )

    if has_readme:
        readme_prs.append({
            "ghid": pr_id,
            "repo": group["repo"].iloc[0],
            "num_files": len(files),
            "files": files,
            "messages": group["msg"].unique().tolist()
        })

print(f"README-related PRs: {len(readme_prs)}")

In [ ]:
# SHOW EXAMPLES

example = readme_prs[0]

print("Repository:")
print(example["repo"])

print("\nFiles changed:")

for f in example["files"]:
    print("-", f)

print("\nMessages:")
print(example["messages"])

In [ ]:
# keep only PRs with README + code files

valid_prs = []

CODE_EXTENSIONS = (
    ".py", ".js", ".java", ".cpp",
    ".c", ".ts", ".go", ".rs"
)

for pr in readme_prs:

    has_code = any(
        f.endswith(CODE_EXTENSIONS)
        for f in pr["files"]
    )

    if has_code:
        valid_prs.append(pr)

print("README + code PRs:", len(valid_prs))

In [7]:
# Find the repository column
repo_candidates = [c for c in df.columns if any(k in c.lower() for k in ['repo', 'project', 'owner', 'name'])]
print('Possible repo columns:', repo_candidates)

REPO_COL = repo_candidates[0] if repo_candidates else None

if REPO_COL:
    print(f'\n=== TOP 20 REPOSITORIES (by number of PRs) ===')
    print(df[REPO_COL].value_counts().head(20))
    print(f'\nTotal unique repos: {df[REPO_COL].nunique():,}')

NameError: name 'df' is not defined

In [ ]:
# Check for labels / outcome columns (was README actually updated?)
label_candidates = [c for c in df.columns if any(k in c.lower()
                    for k in ['label', 'update', 'changed', 'outdated', 'merged', 'status', 'tag'])]
print('Possible label/outcome columns:', label_candidates)

for col in label_candidates:
    print(f'\n--- {col} ---')
    print(df[col].value_counts())

In [ ]:
readme_rows = df[df['file'].str.lower().str.endswith('readme.md')]
print(len(readme_rows))
print(readme_rows[['repo', 'file', 'oldf', 'patch']].head(3))

## 8. Documentation debt signals

In [ ]:
# Keywords that suggest documentation debt
DOC_DEBT_KEYWORDS = [
    'fix doc', 'update doc', 'missing doc', 'outdated', 'undocumented',
    'add doc', 'improve doc', 'documentation', 'readme', 'docstring',
    'javadoc', 'missing comment', 'no comment', 'no description'
]

# Try to find a PR title or description column
title_candidates = [c for c in df.columns if any(k in c.lower()
                    for k in ['title', 'message', 'msg', 'description', 'pr_', 'commit'])]
print('Possible PR title/message columns:', title_candidates)

TITLE_COL = title_candidates[0] if title_candidates else None

if TITLE_COL:
    pattern = '|'.join(DOC_DEBT_KEYWORDS)
    mask = df[TITLE_COL].astype(str).str.lower().str.contains(pattern, na=False)
    print(f'\nPRs with doc debt keywords in title: {mask.sum():,} / {len(df):,} ({mask.mean()*100:.1f}%)')
    print('\n--- Sample doc debt PRs ---')
    print(df[mask][[TITLE_COL]].head(10))

# Conclusions

- The dataset is organized at the file/hunk level rather than the PR level.
- README-related updates can be identified using file path filtering.
- Pull request reconstruction using `ghid` is necessary to reproduce the preprocessing pipeline described in the paper.
- README updates appear to be relatively rare events in the dataset.
- Current analysis only identifies README-related changes and does not yet reconstruct full PR context.

# Planned Pipeline

Raw PR/File Dataset

↓

PR Reconstruction using `ghid`

↓

README-related PR Detection

↓

README + Code Change Filtering

↓

Structured PR-level Dataset

↓

LLM-based README Update Analysis